# Linguistic complexity pipeline: Flesch-Kincaid + Coh-Metrix-lite

Computes two families of complexity/readability metrics for every article in `australia_498sample_climatechange.csv`:

- **Flesch-Kincaid Grade Level, Flesch Reading Ease, Gunning Fog, Coleman-Liau, ARI** -- standard, well-validated readability formulas (via the `textstat` package). These measure *surface reading difficulty* (sentence length, syllables per word), not argumentative complexity.
- **Coh-Metrix-lite** -- an **open-source approximation** of a subset of Coh-Metrix's cohesion/complexity indices (referential cohesion, causal/logical/temporal/additive connective density, mean dependency-tree depth), computed with spaCy. **This is not the official Coh-Metrix tool** -- Coh-Metrix itself has no public API or package, only a web form / desktop app requiring manual per-text submission, so it cannot be automated here. See `src/cohmetrix_lite.py` for exactly what is and isn't approximated, and don't report these as official Coh-Metrix scores in any writeup.

This pipeline is CPU-only -- no LLM, no GPU needed -- so it runs on Colab's free (non-GPU) runtime and finishes quickly even over the full ~480-article corpus.

## 1. Install dependencies

In [ ]:
!pip install -q textstat spacy
!python -m spacy download en_core_web_sm -q
import nltk
nltk.download("cmudict", quiet=True)  # required by textstat for syllable counting

## 2. Clone this repo (pulls the latest complexity-pipeline code and the corpus CSV)

In [ ]:
import os

REPO_URL = "https://github.com/hrauxloh/DAAD_Destructive_polarization"
BRANCH = "claude/concept-language-llama-collab-91hjaq"
REPO_DIR = "/content/DAAD_Destructive_polarization"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} --single-branch {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin {BRANCH}
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} reset --hard origin/{BRANCH}

%cd {REPO_DIR}
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

for name in list(sys.modules):
    if name == "src" or name.startswith("src."):
        del sys.modules[name]

## 3. Run the pipeline over the full corpus

In [ ]:
import csv
import pandas as pd
from src.complexity_pipeline import compute_complexity_table

with open("australia_498sample_climatechange.csv", newline="", encoding="utf-8") as f:
    articles = list(csv.DictReader(f))

print(f"loaded {len(articles)} articles")

rows = compute_complexity_table(articles)
complexity_df = pd.DataFrame(rows)
complexity_df.to_csv("aus_complexity_scores.csv", index=False)
print(f"saved {len(complexity_df)} rows to aus_complexity_scores.csv")
complexity_df.head()

## 4. Quick look at the distribution

In [ ]:
complexity_df.describe()

In [ ]:
from google.colab import files
files.download("aus_complexity_scores.csv")

## Notes / limitations
- **Coh-Metrix-lite is an approximation, not the official tool.** See the docstring in `src/cohmetrix_lite.py` for exactly which indices are approximated and how (lexical-overlap referential cohesion rather than LSA-based, a hand-built connective word list rather than Coh-Metrix's proprietary one, dependency-tree depth rather than Coh-Metrix's specific syntactic indices). If a reviewer needs official Coh-Metrix numbers, texts have to be run through http://tool.cohmetrix.com or the desktop app by hand -- there's no way to script that.
- Flesch-Kincaid and related formulas measure reading difficulty (sentence/word length), not conceptual or argumentative complexity -- keep that distinction explicit in any writeup that also references the propaganda-technique coding pipeline (`notebooks/colab_propaganda_poc.ipynb`), since they're answering different questions about the same texts.
- All of this runs on CPU; no GPU runtime is required for this notebook.